# Phase 2: Daily Pharmacy Demand Forecasting

This notebook implements store-product daily demand forecasting for pharmacy medicines.
Key goals:
- Aggregate transaction lines into a regular daily time series.
- Explicitly fill zero-sale days on the full cartesian grid to maintain temporal regularity.
- Engineer lag, rolling statistics, calendar, and product features without lookahead leakage.
- Validate using chronological splits and expanding-window TimeSeries cross-validation.
- Benchmark baselines against Ridge Regression, Random Forest, and HistGradientBoosting.
- Evaluate with MAE, RMSE, and WAPE.
- Tune hyperparameters on the training timeline and evaluate once on the final untouched test holdout.
- Inspect feature importances and residuals.


In [2]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import joblib

from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.inspection import permutation_importance

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
%matplotlib inline


## 1. Data Ingestion & Full Grid Construction

In real retail time series, days with zero transactions are omitted from raw transaction logs.
We reindex across `store_id x product_id x date` to ensure every day has a defined record (`quantity_sold = 0`) before computing lag and rolling window features.


In [4]:
sales = pd.read_csv('../data/raw/sales_transactions.csv')
products = pd.read_csv('../data/raw/products.csv')
stores = pd.read_csv('../data/raw/stores.csv')

sales['date'] = pd.to_datetime(sales['transaction_datetime']).dt.date
daily_sales = sales.groupby(['store_id', 'product_id', 'date'])['quantity_sold'].sum().reset_index()

full_dates = pd.date_range('2024-01-01', '2025-12-31', freq='D').date
grid = pd.MultiIndex.from_product(
    [stores['store_id'].unique(), products['product_id'].unique(), full_dates],
    names=['store_id', 'product_id', 'date']
)

df = daily_sales.set_index(['store_id', 'product_id', 'date']).reindex(grid, fill_value=0).reset_index()
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(['store_id', 'product_id', 'date']).reset_index(drop=True)

print(f"Total series observations: {len(df):,} (2 stores x 15 products x 731 days)")
print(f"Zero-sale days imputed: {(df['quantity_sold'] == 0).sum():,} ({(df['quantity_sold'] == 0).mean()*100:.2f}%)")
df.head()


Total series observations: 21,930 (2 stores x 15 products x 731 days)
Zero-sale days imputed: 430 (1.96%)


## 2. Leakage-Free Feature Engineering

To prevent lookahead leakage:
- Lags (`lag_1`, `lag_7`, `lag_14`, `lag_28`) use strictly historical values.
- Rolling statistics (mean and std for 7, 14, 28 days) are computed on the `shift(1)` series, ensuring today's target is never part of its own window summary.
- Calendar encodings (day of week, month, cyclical sine/cosine transforms).
- Product catalog features (shelf life, unit price, base daily demand).


In [6]:
grouped = df.groupby(['store_id', 'product_id'])['quantity_sold']

# Historical lags
df['lag_1'] = grouped.shift(1)
df['lag_7'] = grouped.shift(7)
df['lag_14'] = grouped.shift(14)
df['lag_28'] = grouped.shift(28)

# Rolling stats computed strictly on shifted past series
shifted = grouped.shift(1)
df['rolling_mean_7'] = shifted.groupby([df['store_id'], df['product_id']]).transform(lambda x: x.rolling(7).mean())
df['rolling_std_7'] = shifted.groupby([df['store_id'], df['product_id']]).transform(lambda x: x.rolling(7).std())
df['rolling_mean_14'] = shifted.groupby([df['store_id'], df['product_id']]).transform(lambda x: x.rolling(14).mean())
df['rolling_std_14'] = shifted.groupby([df['store_id'], df['product_id']]).transform(lambda x: x.rolling(14).std())
df['rolling_mean_28'] = shifted.groupby([df['store_id'], df['product_id']]).transform(lambda x: x.rolling(28).mean())
df['rolling_std_28'] = shifted.groupby([df['store_id'], df['product_id']]).transform(lambda x: x.rolling(28).std())

# Merge product catalog attributes
df = df.merge(products[['product_id', 'category', 'shelf_life_days', 'base_daily_demand', 'unit_price', 'unit_cost']], on='product_id', how='left')

# Calendar & cyclical features
df['day_of_week'] = df['date'].dt.dayofweek
df['day_of_month'] = df['date'].dt.day
df['month'] = df['date'].dt.month
df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)
df['sin_dow'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
df['cos_dow'] = np.cos(2 * np.pi * df['day_of_week'] / 7)
df['sin_month'] = np.sin(2 * np.pi * (df['month'] - 1) / 12)
df['cos_month'] = np.cos(2 * np.pi * (df['month'] - 1) / 12)

# Drop initial burn-in window (first 28 days have NaN lags)
df_clean = df.dropna().reset_index(drop=True)
df_clean.to_csv('../data/processed/daily_demand_features.csv', index=False)
print(f"Cleaned feature dataset: {df_clean.shape[0]} rows, {df_clean.shape[1]} columns")
print(f"Modeled timeline: {df_clean['date'].min().strftime('%Y-%m-%d')} to {df_clean['date'].max().strftime('%Y-%m-%d')}")


Cleaned feature dataset: 21090 rows, 27 columns
Modeled timeline: 2024-01-29 to 2025-12-31


## 3. Chronological Train / Validation / Test Splitting

Time-series data must never be randomly split. We partition chronologically:
- **Train Set**: 2024-01-29 to 2025-06-30 (518 days, 15,540 rows)
- **Validation Set**: 2025-07-01 to 2025-09-30 (Q3 2025, 92 days, 2,760 rows) — used for model selection
- **Test Set (Holdout)**: 2025-10-01 to 2025-12-31 (Q4 2025, 92 days, 2,760 rows) — strictly held out until final assessment


In [8]:
train_mask = df_clean['date'] <= '2025-06-30'
val_mask = (df_clean['date'] > '2025-06-30') & (df_clean['date'] <= '2025-09-30')
test_mask = df_clean['date'] > '2025-09-30'

train_df = df_clean[train_mask].sort_values('date').reset_index(drop=True)
val_df = df_clean[val_mask].sort_values('date').reset_index(drop=True)
test_df = df_clean[test_mask].sort_values('date').reset_index(drop=True)

cat_cols = ['store_id', 'product_id', 'category']
num_cols = [
    'lag_1', 'lag_7', 'lag_14', 'lag_28',
    'rolling_mean_7', 'rolling_std_7', 'rolling_mean_14', 'rolling_std_14', 'rolling_mean_28', 'rolling_std_28',
    'base_daily_demand', 'shelf_life_days', 'unit_price', 'unit_cost',
    'day_of_week', 'day_of_month', 'month', 'is_weekend',
    'sin_dow', 'cos_dow', 'sin_month', 'cos_month'
]
feature_cols = cat_cols + num_cols
target_col = 'quantity_sold'

X_train, y_train = train_df[feature_cols], train_df[target_col]
X_val, y_val = val_df[feature_cols], val_df[target_col]
X_test, y_test = test_df[feature_cols], test_df[target_col]

print(f"Train set: {len(X_train):,} samples ({train_df['date'].min().date()} to {train_df['date'].max().date()})")
print(f"Val set:   {len(X_val):,} samples ({val_df['date'].min().date()} to {val_df['date'].max().date()})")
print(f"Test set:  {len(X_test):,} samples ({test_df['date'].min().date()} to {test_df['date'].max().date()})")


Train set: 15,570 samples (2024-01-29 to 2025-06-30)
Val set:   2,760 samples (2025-07-01 to 2025-09-30)
Test set:  2,760 samples (2025-10-01 to 2025-12-31)


## 4. Evaluation Metrics & Baselines

We evaluate models using:
- **MAE** (Mean Absolute Error): Average magnitude of forecast error in units sold.
- **RMSE** (Root Mean Squared Error): Penalizes larger errors more heavily.
- **WAPE** (Weighted Absolute Percentage Error): $\frac{\sum |y - \hat{y}|}{\sum y}$. Standard retail KPI resilient to zero-demand instances.


In [10]:
def wape(y_true, y_pred):
    return np.sum(np.abs(y_true - y_pred)) / np.sum(y_true)

def evaluate_predictions(name, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    w = wape(y_true, y_pred)
    return {'Model': name, 'MAE': mae, 'RMSE': rmse, 'WAPE (%)': w * 100}

results = []
results.append(evaluate_predictions('Baseline (7d Rolling Mean)', y_val, val_df['rolling_mean_7']))
results.append(evaluate_predictions('Baseline (Lag 7)', y_val, val_df['lag_7']))
print(pd.DataFrame(results).to_string(index=False))


                     Model      MAE     RMSE  WAPE (%)
Baseline (7d Rolling Mean) 3.660507 4.771882 24.391009
          Baseline (Lag 7) 4.541667 5.951708 30.262427


## 5. Model Comparison on Validation Set

We train three distinct model families:
1. **Ridge Regression** (Linear model with L2 regularization and standard scaling)
2. **HistGradientBoostingRegressor** (Modern histogram-based gradient boosting)
3. **Random Forest Regressor** (Ensemble of bagged decision trees)


In [12]:
# Preprocessors
linear_prep = ColumnTransformer([
    ('cat', OneHotEncoder(drop='first', sparse_output=False), cat_cols),
    ('num', StandardScaler(), num_cols)
])

tree_prep = ColumnTransformer([
    ('cat', OneHotEncoder(drop='first', sparse_output=False), cat_cols)
], remainder='passthrough')

# 1. Ridge Regression
ridge_model = Pipeline([('prep', linear_prep), ('model', Ridge(alpha=10.0))])
ridge_model.fit(X_train, y_train)
results.append(evaluate_predictions('Ridge Regression', y_val, ridge_model.predict(X_val)))

# 2. Random Forest
rf_model = Pipeline([
    ('prep', tree_prep),
    ('model', RandomForestRegressor(n_estimators=100, max_depth=12, random_state=42, n_jobs=1))
])
rf_model.fit(X_train, y_train)
results.append(evaluate_predictions('Random Forest', y_val, rf_model.predict(X_val)))

# 3. HistGradientBoosting
hgb_model = Pipeline([
    ('prep', tree_prep),
    ('model', HistGradientBoostingRegressor(random_state=42))
])
hgb_model.fit(X_train, y_train)
results.append(evaluate_predictions('HistGradientBoosting', y_val, hgb_model.predict(X_val)))

val_summary = pd.DataFrame(results)
print(val_summary.to_string(index=False))


                     Model      MAE     RMSE  WAPE (%)
Baseline (7d Rolling Mean) 3.660507 4.771882 24.391009
          Baseline (Lag 7) 4.541667 5.951708 30.262427
          Ridge Regression 3.395903 4.385308 22.627877
             Random Forest 3.291050 4.230694 21.929212
      HistGradientBoosting 3.272736 4.240123 21.807177


## 6. Expanding-Window TimeSeries CV & Hyperparameter Tuning

HistGradientBoosting achieves the lowest error and highest training efficiency.
We perform 3-fold expanding-window `TimeSeriesSplit` cross-validation strictly on the training set to optimize:
- `learning_rate`
- `max_leaf_nodes`
- `min_samples_leaf`
- `l2_regularization`


In [14]:
tscv = TimeSeriesSplit(n_splits=3)
param_grid = {
    'model__learning_rate': [0.03, 0.05, 0.1],
    'model__max_leaf_nodes': [15, 31],
    'model__min_samples_leaf': [10, 20],
    'model__l2_regularization': [0.0, 1.0]
}

grid_search = GridSearchCV(
    Pipeline([('prep', tree_prep), ('model', HistGradientBoostingRegressor(random_state=42))]),
    param_grid,
    cv=tscv,
    scoring='neg_mean_absolute_error',
    n_jobs=1
)

grid_search.fit(X_train, y_train)
print(f"Optimal Parameters: {grid_search.best_params_}")
print(f"Cross-Validation MAE: {-grid_search.best_score_:.3f}")

best_model = grid_search.best_estimator_
tuned_val_preds = best_model.predict(X_val)
tuned_val_metrics = evaluate_predictions('Tuned HistGradientBoosting (Val)', y_val, tuned_val_preds)
print(pd.DataFrame([tuned_val_metrics]).to_string(index=False))


Optimal Parameters: {'model__l2_regularization': 0.0, 'model__learning_rate': 0.05, 'model__max_leaf_nodes': 31, 'model__min_samples_leaf': 20}
Cross-Validation MAE: 3.413
                           Model     MAE    RMSE  WAPE (%)
Tuned HistGradientBoosting (Val) 3.25863 4.21606 21.713189


## 7. Final Holdout Test Evaluation

With model selection and tuning complete, we evaluate the champion model on the previously untouched Q4 2025 holdout set (2025-10-01 to 2025-12-31).


In [16]:
test_preds = best_model.predict(X_test)
test_metrics = evaluate_predictions('Tuned HistGradientBoosting (Final Test)', y_test, test_preds)
print(pd.DataFrame([test_metrics]).to_string(index=False))


                                  Model      MAE     RMSE  WAPE (%)
Tuned HistGradientBoosting (Final Test) 3.118957 4.013634 20.197369


## 8. Forecast Diagnostics & Residual Analysis

We inspect:
1. Actual vs Predicted daily demand over the test period.
2. Residual distribution (checking for zero-mean and symmetry).
3. Residuals vs Predicted values (checking for heteroscedasticity).


In [18]:
test_eval_df = test_df.copy()
test_eval_df['predicted_demand'] = test_preds
test_eval_df['residual'] = test_eval_df['quantity_sold'] - test_eval_df['predicted_demand']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Actual vs Predicted aggregate daily demand over test timeline
daily_agg = test_eval_df.groupby('date')[['quantity_sold', 'predicted_demand']].sum()
axes[0, 0].plot(daily_agg.index, daily_agg['quantity_sold'], label='Actual Sales', color='#1f77b4', lw=2)
axes[0, 0].plot(daily_agg.index, daily_agg['predicted_demand'], label='Predicted Demand', color='#ff7f0e', lw=2, linestyle='--')
axes[0, 0].set_title('Network-Wide Daily Demand (Actual vs Forecast) — Q4 2025')
axes[0, 0].set_ylabel('Total Units Sold')
axes[0, 0].legend()

# 2. Residual Distribution
sns.histplot(test_eval_df['residual'], kde=True, ax=axes[0, 1], color='#2ca02c', bins=30)
axes[0, 1].axvline(0, color='red', linestyle='--', lw=1.5)
axes[0, 1].set_title(f"Residual Distribution (Mean = {test_eval_df['residual'].mean():.2f}, Std = {test_eval_df['residual'].std():.2f})")
axes[0, 1].set_xlabel('Residual (Actual - Predicted)')

# 3. Residuals vs Predicted
axes[1, 0].scatter(test_eval_df['predicted_demand'], test_eval_df['residual'], alpha=0.3, color='#9467bd', s=15)
axes[1, 0].axhline(0, color='red', linestyle='--', lw=1.5)
axes[1, 0].set_title('Residuals vs Fitted Values')
axes[1, 0].set_xlabel('Predicted Demand')
axes[1, 0].set_ylabel('Residual')

# 4. Product-level Paracetamol ST_01 sample forecast
sample = test_eval_df[(test_eval_df['store_id'] == 'ST_01') & (test_eval_df['product_id'] == 'MED_001')]
axes[1, 1].plot(sample['date'], sample['quantity_sold'], label='Actual', marker='o', markersize=3, color='#1f77b4')
axes[1, 1].plot(sample['date'], sample['predicted_demand'], label='Predicted', marker='x', markersize=3, color='#ff7f0e', linestyle='--')
axes[1, 1].set_title('Paracetamol 500mg at Central Pharmacy (ST_01)')
axes[1, 1].set_ylabel('Units Sold')
axes[1, 1].legend()

plt.tight_layout()
plt.savefig('../models/demand_forecast_diagnostics.png', dpi=150)
plt.close()


## 9. Model Explainability via Permutation Importance

Permutation importance measures the increase in prediction error when a specific feature's values are randomly shuffled, keeping all other features constant.


In [20]:
perm_results = permutation_importance(
    best_model, X_val, y_val,
    n_repeats=5,
    random_state=42,
    scoring='neg_mean_absolute_error',
    n_jobs=1
)

imp_df = pd.DataFrame({
    'Feature': feature_cols,
    'Importance (MAE Degradation)': perm_results.importances_mean
}).sort_values('Importance (MAE Degradation)', ascending=False).reset_index(drop=True)

plt.figure(figsize=(10, 6))
sns.barplot(data=imp_df.head(10), x='Importance (MAE Degradation)', y='Feature', hue='Feature', palette='Blues_r', legend=False)
plt.title('Top 10 Feature Importances (Permutation MAE Impact on Validation Set)')
plt.xlabel('Mean Increase in Absolute Error when Shuffled')
plt.tight_layout()
plt.savefig('../models/demand_forecast_feature_importance.png', dpi=150)
plt.close()

print(imp_df.head(10).to_string(index=False))


          Feature  Importance (MAE Degradation)
  rolling_mean_28                      1.189825
            lag_1                      0.673177
   rolling_mean_7                      0.527293
          cos_dow                      0.148211
base_daily_demand                      0.147573
   rolling_std_28                      0.051370
      day_of_week                      0.042329
          sin_dow                      0.037013
  rolling_mean_14                      0.010459
           lag_28                      0.005604


## 10. Model Serialization

We serialize the trained pipeline to `models/demand_forecast_model.joblib` for deployment in the Streamlit application.


In [22]:
joblib.dump(best_model, '../models/demand_forecast_model.joblib')
print("Model pipeline successfully serialized to models/demand_forecast_model.joblib")


Model pipeline successfully serialized to models/demand_forecast_model.joblib
